In [1]:
import pandas as pd

TRANSACTION_CSV = "train_transaction.csv"
IDENTITY_CSV = "train_identity.csv"

trans_cols = pd.read_csv(TRANSACTION_CSV, nrows=0).columns.tolist()
id_cols = pd.read_csv(IDENTITY_CSV, nrows=0).columns.tolist()

print(f"train_transaction.csv has {len(trans_cols)} columns")
print(trans_cols)
print()
print(f"train_identity.csv has {len(id_cols)} columns")
print(id_cols)

train_transaction.csv has 394 columns
['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63', 'V64', 'V65', 'V66', 'V67', 'V68', 'V69', 'V70', 'V71', 'V72', 'V73', 'V74', 'V75', 'V76',

In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
import hdbscan
import matplotlib.pyplot as plt

# --- shared schema: keep these names in sync with the rest of the team ---
TRANSACTION_CSV = "train_transaction.csv"
IDENTITY_CSV = "train_identity.csv"

ID_COL = "TransactionID"
TARGET_COL = "isFraud"
UID_COL = "entity_uid"

# columns that can be SHARED between transactions -> become graph edges
ENTITY_COLUMNS = ["card1","card2","card3","card4","card5","card6",
                   "addr1","addr2","P_emaildomain","R_emaildomain",
                   "DeviceType","DeviceInfo","id_30","id_31"]

# columns used only to build the pseudo-UID (entity linking)
UID_SOURCE_COLUMNS = ["card1", "addr1", "D1"]

# behavioral/numeric columns -> features for the anomaly detection model
BASE_BEHAVIOR_COLUMNS = ["TransactionAmt","TransactionDT"] + \
    [f"C{i}" for i in range(1,15)] + [f"D{i}" for i in range(2,16)]

SAMPLE_SIZE = 50_000
N_V_COLUMNS_TO_KEEP = 25
RANDOM_SEED = 42

## Load and merge

In [3]:
df_trans = pd.read_csv(TRANSACTION_CSV)
df_id = pd.read_csv(IDENTITY_CSV)
df = df_trans.merge(df_id, on=ID_COL, how="left")
print(df.shape)

(590540, 434)


## pseudo-UID (entity linking)

In [4]:
# NaN if card1/addr1/D1 is missing: we don't want to link two transactions
# just because they share a missing value
complete_mask = df[UID_SOURCE_COLUMNS].notna().all(axis=1)
uid = (df["card1"].astype("Int64").astype(str) + "_" +
       df["addr1"].astype("Int64").astype(str) + "_" +
       df["D1"].round(0).astype("Int64").astype(str))
df[UID_COL] = np.where(complete_mask, uid, np.nan)
print(f"UID coverage: {df[UID_COL].notna().mean():.1%}")

UID coverage: 88.7%


C:\Users\deend\AppData\Local\Temp\ipykernel_6732\1478649032.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[UID_COL] = np.where(complete_mask, uid, np.nan)


## select the most useful V-columns (RandomForest importance, not plain correlation)
In the dataset we don't know what those values are because of anonymization 

In [5]:
v_cols = [c for c in df.columns if c.startswith("V")]
X_v = df[v_cols].fillna(df[v_cols].median(numeric_only=True))

# supervised feature ranking: catches non-linear patterns that a simple
# correlation would miss. Fast even with ~300 columns.
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1, verbose=1)
rf.fit(X_v, df[TARGET_COL])
importance = pd.Series(rf.feature_importances_, index=v_cols).sort_values(ascending=False)

# redundancy filter: skip a candidate if it's almost a copy of one already picked
top_v = []
for col in importance.index:
    if len(top_v) >= N_V_COLUMNS_TO_KEEP:
        break
    if all(abs(df[col].corr(df[picked])) < 0.9 for picked in top_v):
        top_v.append(col)

print(f"Selected {len(top_v)}/{len(v_cols)} V-columns")
print(importance.head(10))

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:   28.4s
[Parallel(n_jobs=-1)]: Done 168 tasks      | elapsed:  2.7min
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:  3.1min finished
c:\Users\deend\Desktop\Coding\Cluster summer school\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\deend\Desktop\Coding\Cluster summer school\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Selected 25/339 V-columns
V307    0.020350
V310    0.016790
V45     0.015947
V258    0.015547
V257    0.015321
V308    0.014496
V201    0.014104
V127    0.013923
V317    0.012272
V314    0.011629
dtype: float64


## final columns + stratified sample

In [6]:
final_cols = [ID_COL, TARGET_COL, UID_COL] + ENTITY_COLUMNS + BASE_BEHAVIOR_COLUMNS + top_v
final_cols = [c for c in final_cols if c in df.columns]
df = df[final_cols]

frac = min(SAMPLE_SIZE, len(df)) / len(df)
parts = [g.sample(frac=frac, random_state=RANDOM_SEED) for _, g in df.groupby(TARGET_COL)]
df = pd.concat(parts, ignore_index=True)
print(df.shape, df[TARGET_COL].mean())

(50000, 72) 0.035


## prepare behavioral features

In [7]:
behavior_cols = [c for c in BASE_BEHAVIOR_COLUMNS if c in df.columns] + top_v
X = df[behavior_cols].fillna(df[behavior_cols].median(numeric_only=True)).fillna(0)
X_scaled = StandardScaler().fit_transform(X)

# Isolation forest + anomaly score

In [8]:
iso = IsolationForest(n_estimators=200, contamination="auto",
                       random_state=RANDOM_SEED, n_jobs=-1)
iso.fit(X_scaled)
raw = -iso.score_samples(X_scaled)  # higher raw = more anomalous
df["anomaly_score"] = (raw - raw.min()) / (raw.max() - raw.min() + 1e-9)

# HDBSCAN -> cluster_id

In [9]:
X_pca = PCA(n_components=10, random_state=RANDOM_SEED).fit_transform(X_scaled)
clusterer = hdbscan.HDBSCAN(min_cluster_size=50, min_samples=10)
df["cluster_id"] = clusterer.fit_predict(X_pca)
print(df["cluster_id"].value_counts().head(10))

cluster_id
-1     28357
 22     4656
 36     4185
 35     3048
 17     1547
 16     1293
 3      1128
 15      472
 42      325
 44      307
Name: count, dtype: int64


## validation against isFraud (sanity check only, not used for training)

In [10]:
auc = roc_auc_score(df[TARGET_COL], df["anomaly_score"])
print(f"AUC anomaly_score vs isFraud: {auc:.3f}")
print(df.groupby("cluster_id")[TARGET_COL].agg(["mean","count"]).sort_values("mean", ascending=False).head(10))

AUC anomaly_score vs isFraud: 0.707
                mean  count
cluster_id                 
4           0.424242     66
2           0.187500     80
13          0.168421     95
8           0.163934     61
30          0.125000     80
7           0.072727     55
11          0.069519    187
14          0.068750    160
19          0.067416     89
24          0.065789     76


In [11]:
cluster4 = df[df["cluster_id"] == 4]
for col in ["card1", "addr1", "DeviceType", "DeviceInfo", "P_emaildomain"]:
    print(col)
    print(cluster4[col].value_counts(dropna=False).head(5))
    print()

card1
card1
6019    7
7585    2
9500    2
1976    2
4461    2
Name: count, dtype: int64

addr1
addr1
NaN      20
264.0     7
204.0     7
299.0     7
327.0     3
Name: count, dtype: int64

DeviceType
DeviceType
desktop    35
mobile     31
Name: count, dtype: int64

DeviceInfo
DeviceInfo
Windows                          22
iOS Device                       18
MacOS                             5
NaN                               5
SAMSUNG SM-G950U Build/NRD90M     2
Name: count, dtype: int64

P_emaildomain
P_emaildomain
gmail.com        35
hotmail.com      12
anonymous.com     4
yahoo.com         3
aol.com           2
Name: count, dtype: int64



In [12]:
for col in ["entity_uid", "card1", "DeviceInfo", "P_emaildomain"]:
    agg = df.groupby(col)[TARGET_COL].agg(fraud_count="sum", n="count", fraud_rate="mean")
    agg = agg[agg["n"] >= 3].sort_values("fraud_count", ascending=False)  # entità viste almeno 3 volte
    print(col)
    print(agg.head(5))
    print()

entity_uid
             fraud_count    n  fraud_rate
entity_uid                               
12501_204_0            5   49    0.102041
1764_315_0             5   20    0.250000
16132_299_0            4  134    0.029851
6019_299_0             4   74    0.054054
15651_330_0            3   34    0.088235

card1
       fraud_count     n  fraud_rate
card1                               
9633            68   362    0.187845
9500            41  1210    0.033884
9917            34    90    0.377778
2616            31   401    0.077307
6019            31   589    0.052632

DeviceInfo
                        fraud_count     n  fraud_rate
DeviceInfo                                           
Windows                         227  4051    0.056036
iOS Device                      115  1689    0.068088
MacOS                            23  1103    0.020852
SM-A300H Build/LRX22G            22    22    1.000000
hi6210sft Build/MRA58K           16    20    0.800000

P_emaildomain
               fraud_cou